## Let's run intron clustering to annotate alternative splicing events given observed junctions in our cells 

In [1]:
!hostname

ne1dc6-008.nygenome.org


In [15]:
import os
import pandas as pd 
from sklearn.decomposition import TruncatedSVD
import anndata as ad
from scipy.sparse import coo_matrix
from datetime import datetime

# turn this into AnnData object 
import anndata as ad
from scipy.sparse import csr_matrix
import numpy as np
import torch 
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

from scipy.spatial.distance import cdist

import numpy as np

import sys
import os
import json
import numpy as np
import torch
import anndata as ad
from importlib import reload
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import pyro 
import umap.umap_ as umap
import matplotlib.patches as mpatches
import scipy.sparse
import datetime
import sys
import random

sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/visualization')
# from visualize_ATSE import visualize_local_events

# Import custom modules
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/beta-dirichlet-factor')

import factor_model
reload(factor_model)

import waypoints_prep_input as wayp
reload(wayp)

# Add path to where find_intron_clusters_v3.py is located 
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/clustering')
from find_intron_clusters_v3 import JunctionReader

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

float_type = {"device": device, "dtype": torch.float}
if device == torch.device('cuda'):
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

!hostname

2.3.0+cu121
12.1
Using device: cpu
ne1dc6-008.nygenome.org


### Specify if we want to prep brain only file or ALL tissue file


In [3]:
brain_only=False
gtf_annot=False

In [4]:
# input_file='/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSE_Anndata_Object_20240927_214100.h5ad' 
# input_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSE_Anndata_noGTF_Object_20250123_232044.h5ad"
input_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSEmap/output/anndata/merged_anndata_compressed.h5ad"

adata_full = ad.read_h5ad(input_file)
splice_adata = adata_full.copy()
splice_adata.obs.reset_index(drop=True, inplace=True)
splice_adata.obs["cell_id_index"] = splice_adata.obs.index 

print(f"The number of cells in the dataset is {splice_adata.shape[0]}")

The number of cells in the dataset is 106199


In [5]:
# Note: this ATSE file was also generated through the script mentioned above
# ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/tabula_senis_test_intron_clusters_50_500000_10_20240927_single_cell.gz"
# ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/tabula_senis_annotationFREE_intron_clusters_50_500000_100_20250123_single_cell.gz"
ATSE_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSEmap/output/ATSEfiles/TMS_atse_file_unanno_also_2025-01-30_19-24-18.txt.gz"
atses = pd.read_csv(ATSE_file, sep="\t")

In [6]:
if brain_only:
    splice_adata = splice_adata[splice_adata.obs["tissue"].isin(["Brain_Myeloid", "Brain_Non-Myeloid"])]
    splice_adata.obs.reset_index(drop=True, inplace=True)
    splice_adata.obs["cell_id_index"] = splice_adata.obs.index 
    print(splice_adata.obs.shape, splice_adata.obs.cell_id_index.max())

### Let's remove the 21m age group since so few cells not well represented... 

In [7]:
splice_adata = splice_adata[~(splice_adata.obs["age"] == "21m")]

##### Clean up Cell IDs in the splicing Anndata object so can merge them with the IDs in the gene expression object

In [8]:
splice_adata.obs["cell_clean"] = splice_adata.obs["cell_id"].str.replace(r'-(?=.*_)', '_', regex=True).values
splice_adata.obs["cell_clean"] = splice_adata.obs["cell_clean"].str.split('_').str[:2].str.join('_')
print(f"The number of cells in the dataset is {splice_adata.shape[0]}")

/scratch/ipykernel_1803006/4257502816.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  splice_adata.obs["cell_clean"] = splice_adata.obs["cell_id"].str.replace(r'-(?=.*_)', '_', regex=True).values
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


The number of cells in the dataset is 106199


### Load in gene expression matrix for smart-seq2 data and align cell IDs 

In [9]:
# Load the expression data (.h5ad file)
exp_file = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/data/AWS/processed_for_scanpy/tabulamurissenisfacsofficialrawobj.h5ad"
adata = sc.read_h5ad(exp_file)

# Reset the index of adata.obs to integers and drop the old index
adata.obs.reset_index(drop=True, inplace=True)
adata.var["mouse_gene_name"] = adata.var.index

# Clean up Cell IDs in the gene expression object
# If age column is 3m, adjust cell_clean to replace the first "." with "_" and the second "." also with "_" (annoying fix but that's how the data came...)
adata.obs['cell_clean'] = adata.obs['cell']  # Start by copying the 'cell' column to 'cell_clean'
adata.obs.loc[adata.obs['age'] == '3m', 'cell_clean'] = adata.obs.loc[adata.obs['age'] == '3m', 'cell'].str.replace('.', '_', 2)
adata.obs["cell_clean"] = adata.obs["cell_clean"].str.split('_').str[:2].str.join('_') 
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

The number of cells in the expression dataset is 110824


### Make Cell IDs comperable between gene expression (downloaded from website) and splicing (made by us) Anndata objects 

In [10]:
# Count how many cells are in the intersection of the two datasets
cell_intersection = set(adata.obs["cell_clean"]).intersection(set(splice_adata.obs["cell_clean"]))
print(len(cell_intersection))

106199


In [11]:
# Make cell IDs comperable 
splice_adata = splice_adata[splice_adata.obs["cell_clean"].isin(adata.obs["cell_clean"])]
adata = adata[adata.obs["cell_clean"].isin(splice_adata.obs["cell_clean"])]
adata.obs.reset_index(drop=True, inplace=True)
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


In [12]:
# Rename the existing 'cell_id_index' to 'old_cell_id_index'
splice_adata.obs.rename(columns={'cell_id_index': 'old_cell_id_index'}, inplace=True)
splice_adata.obs.reset_index(inplace=True, drop=True)
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

# Create a new 'cell_id_index' based on the current index
splice_adata.obs['cell_id_index'] = splice_adata.obs.index
splice_adata.obs_names = splice_adata.obs["cell_clean"]
adata.obs_names = adata.obs["cell_clean"]
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


/scratch/ipykernel_1803006/40729991.py:8: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  splice_adata.obs['cell_id_index'] = splice_adata.obs.index
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [87]:
# junction file with paths to original files 
junction_files = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/junction_files.txt"
junction_files = pd.read_csv(junction_files, sep="\t", header=None)

# Import junction reader class and parser function 
reader = JunctionReader(
        min_intron=50,
        max_intron=500000,
        batch_size=10,
        num_workers=4
    )

In [104]:
# Find all cell-junction pairs with non-zero cluster counts in the sparse matrix 

# Extract the sparse matrix of cell-by-cluster counts
cluster_matrix = splice_adata.layers["cell_by_cluster_matrix"]

# Find indices of non-zero elements
cell_indices, junc_indices = cluster_matrix.nonzero()

# Extract corresponding values (non-zero cluster counts)
nonzero_counts = cluster_matrix[cell_indices, junc_indices].A1  # Convert sparse output to a 1D array

In [154]:
splice_adata.shape

(106199, 270420)

In [153]:
# Find cells where all values in the row are zero
zero_cells_mask = (cluster_matrix.sum(axis=1).A1 == 0)  # A1 flattens sparse output

# Get indices of cells with zero counts
zero_cells = splice_adata.obs.index[zero_cells_mask]

# Convert to DataFrame for easy viewing
zero_cells_df = pd.DataFrame({"cell_id": zero_cells})
zero_cells_df

,cell_id
0,A1_B000633
1,A10_B000795
2,A11_MAA000508
3,A11_MAA000531
4,A11_B000802
...,...
70053,P7_B002847
70054,P7_B003922
70055,P8_B000263
70056,P8_B000808


In [162]:
splice_adata.obs[zero_cells_mask]

,cell_id,age,batch,cell_ontology_class,method,mouse.id,sex,tissue,old_cell_id_index,cell_clean,cell_id_index,subtissue_clean,cell_type_grouped
cell_id_for_index,,,,,,,,,,,,,
A1_B000633,A1-B000633-3_56_F-1-1,3m,1,monocyte,facs,3_56_F,female,Heart,28700,A1_B000633,28700,RV,MONOCYTE
A10_B000795,A10_B000795_B008545_S10,18m,1,"CD4-positive, alpha-beta T cell",facs,18_47_F,female,MAT,28701,A10_B000795,28701,Fat,T CELL
A11_MAA000508,A11-MAA000508-3_9_M-1-1,3m,1,B cell,facs,3_9_M,male,Spleen,28702,A11_MAA000508,28702,NaN,B CELL
A11_MAA000531,A11-MAA000531-3_8_M-1-1,3m,1,macrophage,facs,3_8_M,male,MAT,28703,A11_MAA000531,28703,Fat,MACROPHAGE
A11_B000802,A11_B000802_B009022_S11,18m,1,bulge keratinocyte,facs,18_47_F,female,Skin,28704,A11_B000802,28704,Anagen,KERATINOCYTE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
P7_B002847,P7_B002847_B008550_S367,18m,1,epithelial cell of large intestine,facs,18_46_F,female,Large_Intestine,105131,P7_B002847,105131,Proximal,EPITHELIAL CELL
P7_B003922,P7_B003922_S211_L004,24m,1,endothelial cell of coronary artery,facs,24_59_M,male,Heart,105132,P7_B003922,105132,RV,ENDOTHELIAL CELL
P8_B000263,P8_B000263_B007344_S368,18m,1,macrophage,facs,18_46_F,female,Trachea,105133,P8_B000263,105133,NaN,MACROPHAGE


In [111]:
# Convert to DataFrame for easy viewing
nonzero_df = pd.DataFrame({
    "cell_indices": cell_indices,
    "junc_indices": junc_indices,
    "cluster_count": nonzero_counts
})

nonzero_df.tail()

,cell_indices,junc_indices,cluster_count
1137855704,106198,270405,54.0
1137855705,106198,270406,54.0
1137855706,106198,270407,54.0
1137855707,106198,270408,54.0
1137855708,106198,270409,54.0


In [179]:
# how many unique cell_ids are in nonzero_df
len(nonzero_df["cell_indices"].unique())

36141

In [112]:
# Sample a random cell and a random junction from a row in nonzero_df
random_row = random.choice(nonzero_df.index)
random_cell = nonzero_df.loc[random_row, "cell_indices"]
random_junc = nonzero_df.loc[random_row, "junc_indices"]

random_cell_id = splice_adata.obs[splice_adata.obs["cell_id_index"] == random_cell]["cell_clean"].values[0]

# Get row index of the cell and column index of the junction
cell_idx = random_cell
junc_idx = random_junc

# Retrieve the event ID corresponding to the sampled junction
event_id = splice_adata.var[splice_adata.var["junction_id_index"] == junc_idx]["event_id"].values[0]

# Get all junctions in the same event
event_juncs = splice_adata.var[splice_adata.var["event_id"] == event_id]["junction_id_index"].values
event_juncs_ids = splice_adata.var[splice_adata.var["event_id"] == event_id]["junction_id"].values

# Retrieve counts of all junctions for the selected cell
junction_counts = splice_adata.layers["cell_by_junction_matrix"][cell_idx, event_juncs].toarray().flatten()

# Retrieve cluster counts for the junctions
cluster_counts = splice_adata.layers["cell_by_cluster_matrix"][cell_idx, event_juncs].toarray().flatten()

# Store results in a dictionary
event_counts_dict = {event_juncs[i]: {"junction_count": junction_counts[i], "cluster_count": cluster_counts[i]} 
                     for i in range(len(event_juncs))}

# Print summary
print(f"Random cell: {random_cell}")
print(f"Random cell ID: {random_cell_id}")
print(f"Random junction: {random_junc}")
print(f"Event ID: {event_id}")
print(f"Event junctions: {event_juncs}")
print(f"Event junction IDs: {event_juncs_ids}")

# Print all junction counts and cluster counts
for junc, counts in event_counts_dict.items():
    print(f"Junction: {junc} | Count in Cell: {counts['junction_count']} | Cluster Count: {counts['cluster_count']}")

Random cell: 18050
Random cell ID: P2_B000794
Random junction: 147079
Event ID: ATSE_18365
Event junctions: [147078 147079 147080 147081 147082]
Event junction IDs: ['chr2_5951397_5957470_+' 'chr2_5951553_5952427_+'
 'chr2_5951553_5952434_+' 'chr2_5951553_5957470_+'
 'chr2_5952621_5957470_+']
Junction: 147078 | Count in Cell: 0.0 | Cluster Count: 10.0
Junction: 147079 | Count in Cell: 0.0 | Cluster Count: 10.0
Junction: 147080 | Count in Cell: 0.0 | Cluster Count: 10.0
Junction: 147081 | Count in Cell: 10.0 | Cluster Count: 10.0
Junction: 147082 | Count in Cell: 0.0 | Cluster Count: 10.0


In [168]:
junction_files.iloc[100].values

array(['/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/3_month/J8-MAA001844-3_38_F-1-1/junctions_with_barcodes.bed'],
      dtype=object)

In [177]:
# which cell index corresponds to P9_B001183 
splice_adata.obs[splice_adata.obs["cell_clean"] == "P9_B001183"].old_cell_id_index

cell_id_for_index
P9_B001183    105135
Name: old_cell_id_index, dtype: int64

In [175]:
# which row in junction_files has "J10_B001397" in it 
junc_file = junction_files[junction_files[0].str.contains("P9_B001183")][0].values[0]
print(junc_file)
test_juncs = reader.parse_file(junc_file)
test_juncs

/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/junctions/24_month/P9_B001183_S9_L003/junctions_with_barcodes.bed


{'chr1_4857976_4867469_+': {'cells': 1,
  'total_score': 12,
  'chrom': 'chr1',
  'start': 4857976,
  'end': 4867469,
  'strand': '+'},
 'chr1_4867532_4878026_+': {'cells': 1,
  'total_score': 29,
  'chrom': 'chr1',
  'start': 4867532,
  'end': 4878026,
  'strand': '+'},
 'chr1_4878132_4886743_+': {'cells': 1,
  'total_score': 222,
  'chrom': 'chr1',
  'start': 4878132,
  'end': 4886743,
  'strand': '+'},
 'chr1_4886831_4889459_+': {'cells': 1,
  'total_score': 36,
  'chrom': 'chr1',
  'start': 4886831,
  'end': 4889459,
  'strand': '+'},
 'chr1_4889602_4890739_+': {'cells': 1,
  'total_score': 56,
  'chrom': 'chr1',
  'start': 4889602,
  'end': 4890739,
  'strand': '+'},
 'chr1_4890796_4891914_+': {'cells': 1,
  'total_score': 150,
  'chrom': 'chr1',
  'start': 4890796,
  'end': 4891914,
  'strand': '+'},
 'chr1_4892069_4893416_+': {'cells': 1,
  'total_score': 22,
  'chrom': 'chr1',
  'start': 4892069,
  'end': 4893416,
  'strand': '+'},
 'chr1_4893563_4894933_+': {'cells': 1,
  'tot

### Clean up tissue and cell_type grouping names

In [115]:
# Dictionary to consolidate duplicates
subtissue_corrections = {
    'T cells': 'T-cells',
    'ENDOMUCIN': 'Endomucin',
    'forelimb and hindlimb': 'ForelimbandHindlimb',
    'Liver non-hepato/SCs_st': 'Liver non-hepato/SCs',
    'Skin Anagen': 'Anagen'
}

cell_type_groupings = {
    
    # Basal cells
    'basal cell of epidermis': 'BASAL CELL',
    'basal cell': 'BASAL CELL',

    # Endothelial cells
    'endothelial cell': 'ENDOTHELIAL CELL',
    'endothelial cell of coronary artery': 'ENDOTHELIAL CELL',
    'endothelial cell of hepatic sinusoid': 'ENDOTHELIAL CELL',
    'aortic endothelial cell': 'ENDOTHELIAL CELL',
    'vein endothelial cell': 'ENDOTHELIAL CELL',
    'endothelial cell of lymphatic vessel': 'ENDOTHELIAL CELL',

    # T cells
    'T cell': 'T CELL',
    'CD4-positive, alpha-beta T cell': 'T CELL',
    'CD8-positive, alpha-beta T cell': 'T CELL',
    'regulatory T cell': 'T CELL',
    'mature NK T cell': 'T CELL',
    'mature alpha-beta T cell': 'T CELL',
    
    # B cells
    'B cell': 'B CELL',
    'immature B cell': 'B CELL',
    'naive B cell': 'B CELL',
    'precursor B cell': 'B CELL',
    'early pro-B cell': 'B CELL',
    'late pro-B cell': 'B CELL',
    'plasma cell': 'B CELL',

    # Fibroblasts
    'fibroblast': 'FIBROBLAST',
    'fibroblast of cardiac tissue': 'FIBROBLAST',
    'fibroblast of lung': 'FIBROBLAST',
    'pulmonary interstitial fibroblast': 'FIBROBLAST',
    'kidney interstitial fibroblast': 'FIBROBLAST',
    'fibrocyte': 'FIBROBLAST',

    # Macrophages 
    'macrophage': 'MACROPHAGE',
    'Kupffer cell': 'MACROPHAGE', # macrophages in the liver
    'lung macrophage': 'MACROPHAGE',

    # Monocytes
    'monocyte': 'MONOCYTE',
    'classical monocyte': 'MONOCYTE',
    'non-classical monocyte': 'MONOCYTE',
    'intermediate monocyte': 'MONOCYTE',

    # General Immune Cells
    'granulocyte': 'GRANULOCYTE',
    'basophil': 'GRANULOCYTE', 
    'granulocyte monocyte progenitor cell': 'GRANULOCYTE',

    'leukocyte': 'GENERAL IMMUNE CELL',
    'professional antigen presenting cell': 'ANTIGEN PRESENTING CELL',

    'lymphocyte': 'LYMPHOID IMMUNE CELL',
    'NK cell': 'LYMPHOID IMMUNE CELL',

    'myeloid cell': 'MYELOID IMMUNE CELL',
    'myeloid leukocyte': 'MYELOID IMMUNE CELL',
    'granulocytopoietic cell': 'MYELOID IMMUNE CELL',
    'promonocyte': 'MYELOID IMMUNE CELL',

    'thymocyte': 'THYMOCYTE',
    'DN4 thymocyte': 'THYMOCYTE',

    # Neutrophils
    'neutrophil': 'NEUTROPHIL',

    # Dendritic Cells
    'dendritic cell': 'DENDRITIC CELL',
    'plasmacytoid dendritic cell': 'DENDRITIC CELL',
    'myeloid dendritic cell': 'DENDRITIC CELL',

    # Microglia (Brain Immune Cells)
    'microglial cell': 'MICROGLIA',
    
    # Pancreatic cells
    'pancreatic A cell': 'PANCREATIC CELL',
    'pancreatic B cell': 'PANCREATIC CELL',
    'pancreatic D cell': 'PANCREATIC CELL',
    'pancreatic acinar cell': 'PANCREATIC CELL',
    'pancreatic PP cell': 'PANCREATIC CELL',
    'pancreatic ductal cell': 'PANCREATIC CELL',
    'pancreatic stellate cell': 'PANCREATIC CELL',
    
    # Smooth muscle cells
    'smooth muscle cell': 'SMOOTH MUSCLE CELL',
    'bronchial smooth muscle cell': 'SMOOTH MUSCLE CELL',
    'smooth muscle cell of the pulmonary artery': 'SMOOTH MUSCLE CELL',
    'smooth muscle cell of trachea': 'SMOOTH MUSCLE CELL',
    
    # Epithelial cells (includes luminal epithelial cell of mammary gland)
    'epithelial cell': 'EPITHELIAL CELL',
    'epidermal cell': 'EPITHELIAL CELL',
    'epithelial cell of large intestine': 'EPITHELIAL CELL',
    'enterocyte of epithelium of large intestine': 'EPITHELIAL CELL',
    'epithelial cell of proximal tubule': 'EPITHELIAL CELL',
    'epithelial cell of thymus': 'EPITHELIAL CELL',
    'bladder urothelial cell': 'EPITHELIAL CELL',
    'basal epithelial cell of tracheobronchial tree': 'EPITHELIAL CELL',
    'luminal epithelial cell of mammary gland': 'EPITHELIAL CELL',

    # Neurons
    'neuron': 'NEURON',
    'medium spiny neuron': 'NEURON',
    'interneuron': 'NEURON',
    'neuronal stem cell': 'NEURON',

    # Glial Cells (excluding microglia)
    'oligodendrocyte': 'GLIAL CELL',
    'oligodendrocyte precursor cell': 'GLIAL CELL',
    'astrocyte': 'GLIAL CELL',
    'Bergmann glial cell': 'GLIAL CELL',
    'ependymal cell': 'GLIAL CELL',

    # Stem cells
    'mesenchymal stem cell': 'STEM CELL',
    'mesenchymal stem cell of adipose': 'STEM CELL',
    'hematopoietic stem cell': 'STEM CELL',
    'neuronal stem cell': 'STEM CELL',
    'intestinal crypt stem cell': 'STEM CELL',
    'keratinocyte stem cell': 'STEM CELL',
    'lymphoid progenitor cell': 'STEM CELL',
    'proerythroblast': 'STEM CELL',
    'megakaryocyte-erythroid progenitor cell': 'STEM CELL',

    # Other specialized cells
    'ventricular myocyte': 'CARDIAC MUSCLE CELL',
    'atrial myocyte': 'CARDIAC MUSCLE CELL',
    'skeletal muscle satellite cell': 'SKELETAL MUSCLE CELL',
    'kidney collecting duct principal cell': 'KIDNEY CELL',
    'kidney collecting duct epithelial cell': 'KIDNEY CELL',
    'kidney interstitial fibroblast': 'FIBROBLAST',
    'mesangial cell': 'KIDNEY CELL',
    'type I pneumocyte': 'LUNG CELL',
    'type II pneumocyte': 'LUNG CELL',
    'club cell of bronchiole': 'LUNG CELL',
    'lung neuroendocrine cell': 'LUNG CELL',
    'ciliated columnar cell of tracheobronchial tree': 'LUNG CELL',
    'respiratory basal cell': 'LUNG CELL',
    'Brush cell of epithelium proper of large intestine': 'INTESTINAL CELL',
    'large intestine goblet cell': 'INTESTINAL CELL',
    'enteroendocrine cell': 'INTESTINAL CELL',
    'stromal cell': 'STROMAL CELL',
    'pericyte cell': 'PERICYTE',
    'brain pericyte': 'PERICYTE',
    'adventitial cell': 'STROMAL CELL',
    'keratinocyte': 'KERATINOCYTE',
    'bulge keratinocyte': 'KERATINOCYTE',
    'hepatocyte': 'HEPATOCYTE',
    'bladder cell': 'BLADDER CELL',
    'secretory cell': 'SECRETORY CELL',
    'endocardial cell': 'ENDOCARDIAL CELL',
    'valve cell': 'VALVE CELL',
    'chondrocyte': 'STROMAL CELL',
    'fenestrated cell': 'FENESTRATED CELL',
    'neuroepithelial cell': 'NEUROEPITHELIAL CELL',
    'kidney loop of Henle ascending limb epithelial cell': 'KIDNEY CELL',
    'mucus secreting cell': 'SECRETORY CELL'
}

In [116]:
# Remove any leading/trailing whitespace from subtissue values
splice_adata.obs['subtissue'] = splice_adata.obs['subtissue'].str.strip()
splice_adata.obs['subtissue_clean'] = splice_adata.obs['subtissue'].replace(subtissue_corrections)

# Drop the old subtissue 
splice_adata.obs.drop(columns=['subtissue'], inplace=True)

# Add new cell type groupings 
splice_adata.obs['cell_ontology_class'] = splice_adata.obs['cell_ontology_class'].astype(str)
splice_adata.obs['cell_type_grouped'] = splice_adata.obs['cell_ontology_class'].replace(cell_type_groupings)
splice_adata.obs['cell_type_grouped'] = splice_adata.obs['cell_type_grouped'].fillna(adata.obs['cell_ontology_class'])

In [117]:
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


### Subset to just cell types with more than 50 cells in them

In [118]:
# Step 1: Count the number of cells per cell type in 'cell_ontology_class'
cell_type_counts = splice_adata.obs['cell_type_grouped'].value_counts()

# Step 2: Filter for cell types with more than 50 cells
cell_types_to_keep = cell_type_counts[cell_type_counts > 50].index

# Step 3: Subset the AnnData object to only include these cell types
splice_adata = splice_adata[splice_adata.obs['cell_type_grouped'].isin(cell_types_to_keep)]

# Print the subsetted cell types and their counts
print(splice_adata.obs['cell_type_grouped'].value_counts())
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata.shape[0]}")

cell_type_grouped
MICROGLIA                  12796
STEM CELL                  10946
B CELL                     10221
ENDOTHELIAL CELL            8697
EPITHELIAL CELL             7715
FIBROBLAST                  5715
BASAL CELL                  5334
MYELOID IMMUNE CELL         4106
T CELL                      4063
KERATINOCYTE                3718
THYMOCYTE                   3327
GRANULOCYTE                 3222
PANCREATIC CELL             3004
GLIAL CELL                  2937
SKELETAL MUSCLE CELL        2682
MACROPHAGE                  2678
SMOOTH MUSCLE CELL          2274
INTESTINAL CELL             1886
MONOCYTE                    1602
HEPATOCYTE                  1152
BLADDER CELL                 939
LYMPHOID IMMUNE CELL         855
STROMAL CELL                 823
KIDNEY CELL                  805
NEURON                       794
SECRETORY CELL               589
CARDIAC MUSCLE CELL          542
PERICYTE                     519
DENDRITIC CELL               441
ENDOCARDIAL CELL         

In [119]:
# Filter the gene expression object to just these cells and add the useful columns 
adata_reordered = adata[splice_adata.obs_names]
splice_adata.obs.rename_axis('cell_id_for_index', inplace=True)
adata_reordered.obs.rename_axis('cell_id_for_index', inplace=True)

original_order = adata_reordered.obs.index
merged_obs = adata_reordered.obs.merge(splice_adata.obs, on=['cell_clean', 'age', 'method', 'mouse.id', 'cell_ontology_class', 'tissue', 'sex'])
merged_obs.set_index(original_order, inplace=True)
adata_reordered.obs = merged_obs

print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


### Sanity check to ensure cells ordered in the same way between the two objects!

In [120]:
(adata_reordered.obs_names == splice_adata.obs_names).all()

True

### Processing gene expression data --> raw counts --> normalize

In [ ]:
# Save raw counts as adata_reordered layer 
adata_reordered.layers["raw_counts"] = adata_reordered.X.copy()
adata_reordered.layers["raw_counts"].data
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

# Remove low quality cells and genes
sc.pp.filter_cells(adata_reordered, min_genes=200)
sc.pp.filter_genes(adata_reordered, min_cells=3)

# Step 1: Normalize the data and log transform it
sc.pp.normalize_total(adata_reordered, target_sum=1e4)
sc.pp.log1p(adata_reordered)

# Step 2: Identify highly variable genes
sc.pp.highly_variable_genes(adata_reordered, n_top_genes=2000, subset=False)  
print(f"Number of highly variable genes: {adata_reordered.shape[1]}")

# Step 3: Perform PCA
sc.pp.pca(adata_reordered, n_comps=50)  # Default is 50 principal components
print("PCA completed. Shape of the PCA matrix:", adata_reordered.obsm['X_pca'].shape)

# Step 4: Compute the neighborhood graph (needed for UMAP and clustering)
sc.pp.neighbors(adata_reordered, n_neighbors=10, n_pcs=40)  # Use 40 PCs

# Step 5: Perform UMAP
sc.tl.umap(adata_reordered)
print("UMAP completed. Shape of the UMAP embedding:", adata_reordered.obsm['X_umap'].shape)

# Step 6: Perform clustering (Leiden algorithm by default)
sc.tl.leiden(adata_reordered, resolution=1.0)  # Adjust the resolution to control cluster granularity
print("Leiden clustering completed. Number of clusters:", adata_reordered.obs['leiden'].nunique())

### Merge ATSE gene information if it's there to splice_adata object!

In [121]:
gtf_annot=True

if gtf_annot:
    # find common columns between splice_adata.var and atses
    splice_adata.var = splice_adata.var.merge(atses[["gene_id", "gene_name", "junction_id", "annotation_status", "position_off_5_prime", "position_off_3_prime"]], on=["junction_id", "gene_id"])
else:
    splice_adata.var = splice_adata.var.merge(atses[["junction_id"]], on=["junction_id"])    

### Obtain sparse junction usage ratios! (First filter to keep only junctions with reasonable coverage...)

In [122]:
splice_adata.var.head()

,junction_id,gene_id,event_id,CountJuncs,splice_motif,label_5_prime,label_3_prime,junction_id_index,gene_name,annotation_status,position_off_5_prime,position_off_3_prime
0,chr10_100016035_100021457_+,ENSMUSG00000019966.18,ATSE_23615,132,GT-AG,annotated on 5',unannotated on 3',0,Kitl,five_prime,0.0,NaN
1,chr10_100016035_100051845_+,ENSMUSG00000019966.18,ATSE_23615,191547,GT-AG,annotated on 5',annotated on 3',1,Kitl,both,0.0,1.0
2,chr10_100016035_100064083_+,ENSMUSG00000019966.18,ATSE_23615,370,GT-AG,annotated on 5',annotated on 3',2,Kitl,both,0.0,1.0
3,chr10_100016035_100087346_+,ENSMUSG00000019966.18,ATSE_23615,266,GT-AG,annotated on 5',annotated on 3',3,Kitl,both,0.0,1.0
4,chr10_100016155_100051845_+,ENSMUSG00000019966.18,ATSE_23615,3734,GT-AG,annotated on 5',annotated on 3',4,Kitl,both,0.0,1.0


In [152]:
splice_adata

AnnData object with n_obs × n_vars = 106199 × 270420
    obs: 'cell_id', 'age', 'batch', 'cell_ontology_class', 'method', 'mouse.id', 'sex', 'tissue', 'old_cell_id_index', 'cell_clean', 'cell_id_index', 'subtissue_clean', 'cell_type_grouped'
    var: 'junction_id', 'gene_id', 'event_id', 'CountJuncs', 'splice_motif', 'label_5_prime', 'label_3_prime', 'junction_id_index', 'gene_name', 'annotation_status', 'position_off_5_prime', 'position_off_3_prime'
    layers: 'cell_by_cluster_matrix', 'cell_by_junction_matrix'

In [123]:
print(splice_adata.var.splice_motif.value_counts())
print(splice_adata.var.annotation_status.value_counts())

splice_motif
GT-AG    264310
GC-AG      6021
AT-AC        89
Name: count, dtype: int64
annotation_status
both           144817
five_prime      54002
three_prime     48853
unannotated     22748
Name: count, dtype: int64


In [134]:
# find event_id in which all junctions have "both" under annotation_status
event_ids = splice_adata.var.groupby("event_id")["annotation_status"].value_counts().unstack().fillna(0)
event_ids = event_ids[event_ids["both"] == event_ids.sum(axis=1)]
event_ids.index.values
print(f"Number of ATSE event IDs corresponding to junctions with 'both' annotation status: {len(event_ids)}")

Number of ATSE event IDs corresponding to junctions with 'both' annotation status: 11420


/scratch/ipykernel_1803006/1659040527.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  event_ids = splice_adata.var.groupby("event_id")["annotation_status"].value_counts().unstack().fillna(0)


In [133]:
# Let's use just those for the waypoint analysis 
splice_adata_subset = splice_adata[:, splice_adata.var["event_id"].isin(event_ids.index.values)].copy()
print(f"The number of cells and junctions in the subsetted splicing dataset is {splice_adata_subset.shape}")

Number of ATSE event IDs corresponding to junctions with 'both' annotation status: 49362
The number of cells and junctions in the subsetted splicing dataset is (106199, 35723)


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [137]:
# Extract junction counts and cluster counts as sparse matrices
junction_counts = splice_adata_subset.layers["cell_by_junction_matrix"] 
junction_counts = junction_counts.tocoo()

cluster_counts = splice_adata_subset.layers["cell_by_cluster_matrix"]    
cluster_counts = cluster_counts.tocoo()

In [139]:
# Filter junctions, ensuring they have reasonable coverage 
junc_norm_sum = wayp.sparse_sum(junction_counts, 0) / junction_counts.shape[0]
min_junc_mean = 0.005 # default 
to_keep = junc_norm_sum > min_junc_mean
juncs_filter = splice_adata_subset.var[to_keep]

print(f"Number of junctions in matrix before filtering: {len(junc_norm_sum)}")
print(f"Number of junctions after filtering {len(junc_norm_sum[to_keep])}")

Number of junctions in matrix before filtering: 35723
Number of junctions after filtering 24266


### Ensure we are only keeping ATSEs for downstream analysis with at least two junctions post filtering 

In [143]:
cluster_junction_counts = juncs_filter.groupby("event_id")["junction_id"].count()
clusters_to_keep = cluster_junction_counts[cluster_junction_counts >= 2].index
juncs_filter_final = juncs_filter[juncs_filter["event_id"].isin(clusters_to_keep)]

/scratch/ipykernel_1803006/753007849.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cluster_junction_counts = juncs_filter.groupby("event_id")["junction_id"].count()


In [144]:
splice_adata_subset.var['old_index'] = splice_adata_subset.var.index
splice_adata_subset = splice_adata_subset[:, juncs_filter_final.index]
splice_adata_subset.var = splice_adata_subset.var.reset_index(drop=True)
splice_adata_subset.var["junction_id_index"] = splice_adata_subset.var.index

print(f"The number of cells in the splicing dataset is {splice_adata_subset.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

The number of cells in the splicing dataset is 106199
The number of cells in the expression dataset is 106199


In [145]:
splice_adata_subset

AnnData object with n_obs × n_vars = 106199 × 22084
    obs: 'cell_id', 'age', 'batch', 'cell_ontology_class', 'method', 'mouse.id', 'sex', 'tissue', 'old_cell_id_index', 'cell_clean', 'cell_id_index', 'subtissue_clean', 'cell_type_grouped'
    var: 'junction_id', 'gene_id', 'event_id', 'CountJuncs', 'splice_motif', 'label_5_prime', 'label_3_prime', 'junction_id_index', 'gene_name', 'annotation_status', 'position_off_5_prime', 'position_off_3_prime', 'old_index'
    layers: 'cell_by_cluster_matrix', 'cell_by_junction_matrix'

### Now make sure there are no cells wtih zero counts across the board!

In [146]:
# Access the sparse matrix with junction counts (assumed to be in layers['cell_by_junction_matrix'])
junction_matrix = splice_adata.layers['cell_by_junction_matrix']

# Check if the matrix is sparse (it should be sparse to avoid conversion to dense)
if isinstance(junction_matrix, csr_matrix):
    # Sum across each row (axis=1) and check if the sum is zero (meaning the row is all zeros)
    non_zero_cells = junction_matrix.getnnz(axis=1) > 0
else:
    raise ValueError("The matrix is not sparse!")

print(len(non_zero_cells))

106199


In [149]:
# Check how many True / False in non_zero_cells
print(f"Number of cells with non-zero junction counts: {non_zero_cells.sum()}")
print(f"Number of cells with zero junction counts: {(~non_zero_cells).sum()}")

Number of cells with non-zero junction counts: 36141
Number of cells with zero junction counts: 70058


In [151]:
# Which cells have zero junction counts?
zero_cells = splice_adata_subset.obs[~non_zero_cells]
zero_cells

,cell_id,age,batch,cell_ontology_class,method,mouse.id,sex,tissue,old_cell_id_index,cell_clean,cell_id_index,subtissue_clean,cell_type_grouped
cell_id_for_index,,,,,,,,,,,,,
A1_B000633,A1-B000633-3_56_F-1-1,3m,1,monocyte,facs,3_56_F,female,Heart,28700,A1_B000633,28700,RV,MONOCYTE
A10_B000795,A10_B000795_B008545_S10,18m,1,"CD4-positive, alpha-beta T cell",facs,18_47_F,female,MAT,28701,A10_B000795,28701,Fat,T CELL
A11_MAA000508,A11-MAA000508-3_9_M-1-1,3m,1,B cell,facs,3_9_M,male,Spleen,28702,A11_MAA000508,28702,NaN,B CELL
A11_MAA000531,A11-MAA000531-3_8_M-1-1,3m,1,macrophage,facs,3_8_M,male,MAT,28703,A11_MAA000531,28703,Fat,MACROPHAGE
A11_B000802,A11_B000802_B009022_S11,18m,1,bulge keratinocyte,facs,18_47_F,female,Skin,28704,A11_B000802,28704,Anagen,KERATINOCYTE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
P7_B002847,P7_B002847_B008550_S367,18m,1,epithelial cell of large intestine,facs,18_46_F,female,Large_Intestine,105131,P7_B002847,105131,Proximal,EPITHELIAL CELL
P7_B003922,P7_B003922_S211_L004,24m,1,endothelial cell of coronary artery,facs,24_59_M,male,Heart,105132,P7_B003922,105132,RV,ENDOTHELIAL CELL
P8_B000263,P8_B000263_B007344_S368,18m,1,macrophage,facs,18_46_F,female,Trachea,105133,P8_B000263,105133,NaN,MACROPHAGE


In [ ]:
# Filter the AnnData object to remove cells with all zero counts
# splice_adata = splice_adata[non_zero_cells, :]
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

#### Get sparse PCA, UMAP and Diffusion Map!

In [ ]:
# Extract junction counts and cluster counts as sparse matrices 
junction_counts = splice_adata.layers["cell_by_junction_matrix"] 
cluster_counts = splice_adata.layers["cell_by_cluster_matrix"]    

# Ensure the matrices are in COO format for easier element-wise operations
junction_counts = junction_counts.tocoo()
cluster_counts = cluster_counts.tocoo()

In [ ]:
# Get sparse centered PSI values 
splice_adata.layers["junc_ratio"] = wayp.calculate_centered_psi(junction_counts, cluster_counts)

# Step 1: Perform PCA using sparse data
n_components = 40  # Number of components to keep

n_iter = 5  # Increasing the number of iterations for better convergence
svd = TruncatedSVD(n_components=n_components, n_iter=n_iter, random_state=42)

# Fit and transform the junction ratio data (this gives U)
U = svd.fit_transform(splice_adata.layers["junc_ratio"])

# Get the singular values (S)
S = svd.singular_values_

# Multiply U by S to get U * S
# Need to scale U by the singular values S (broadcasted across columns)
U_by_S = U * S  # This scales each component in U by the corresponding singular value in S

# Store the PCA results (U * S) in the 'X_pca' field of .obsm (multi-dimensional)
splice_adata.obsm['X_pca'] = U_by_S

# Optionally, store explained variance ratio for future reference
splice_adata.uns['pca_explained_variance_ratio'] = svd.explained_variance_ratio_

In [ ]:
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

In [ ]:
pca_result = U_by_S

# Step 2: Compute UMAP on the PCA-reduced data
sc.pp.neighbors(splice_adata, use_rep='X_pca')

# Step 3. Calculate UMAP 
sc.tl.umap(splice_adata)

# Step 4. Calculate the diffusion map
sc.tl.diffmap(splice_adata)

# The diffusion components are now stored in adata.obsm['X_diffmap']
diffmap_components = splice_adata.obsm['X_diffmap']
pca_components = splice_adata.obsm["X_pca"]

# Step 5. Run tSNE 
sc.tl.tsne(splice_adata)
tsne_components = splice_adata.obsm["X_tsne"]

In [ ]:
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

### Identify cell waypoints using splicing based PCs!

In [ ]:
# Define possible number of waypoints to learn
n_waypoints_learn = [30, 50, 100]

# Placeholder to store waypoints and metacell dictionaries for each n_waypoints
waypoints_dict = {}
metacell_dicts = {}

# Parameters
num_components = 20  # Number of diffusion map components to consider
metacell_size = 75    # Number of nearest cells to assign to each waypoint

# Loop over different n_waypoints to generate waypoints and metacell assignments
for n_waypoints in n_waypoints_learn:

    print(f"Finding {n_waypoints} waypoints from the diffusion components!")
    random_seed = np.random.randint(0, 10000 + 1)  # Generate random seed
    
    # Max-min sampling to identify waypoints
    waypoints = wayp.max_min_sampling(pca_components, n_waypoints, num_components=num_components, seed=random_seed)
    
    # Store waypoints for this particular number of waypoints
    waypoints_dict[n_waypoints] = waypoints

    # Assign nearest cells to each waypoint (metacells)
    metacell_dict = wayp.assign_nearest_cells(waypoints, pca_components, num_nearest=metacell_size)
    
    # Store the metacell dictionary for this number of waypoints
    metacell_dicts[n_waypoints] = metacell_dict

#### Try looking at waypoints just using PC space...

In [ ]:
wayp.plot_PCA_with_waypoints(splice_adata, waypoints_dict, n_waypoints=30, waypoint_color='red', first_waypoint_color='blue', size=10)

# wayp.plot_tSNE_with_waypoints(splice_adata, waypoints_dict, n_waypoints=50, waypoint_color='red', first_waypoint_color='blue', size=10)
# wayp.plot_UMAP_with_waypoints(splice_adata, waypoints_dict, n_waypoints=50, waypoint_color='red', first_waypoint_color='blue', size=10)

In [ ]:
# Get the unique tissues in adata.obs['cell_ontology_class']
tissues = splice_adata.obs['cell_type_grouped'].unique()

# Set up grid dimensions: 6 rows, adjust columns to fit the number of tissues
n_tissues = len(tissues)
n_rows = 4  # Set to 6 rows
n_cols = int(np.ceil(n_tissues / n_rows))  # Calculate the number of columns needed

# Create a grid of subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))  # Adjust figure size as needed
axes = axes.flatten()  # Flatten axes to easily loop through them

# Plot each tissue's UMAP in its subplot without legend
for i, tissue in enumerate(tissues):
    ax = axes[i]
    
    # Plot UMAP for the specific tissue on the corresponding subplot without the legend
    sc.pl.umap(splice_adata, color="cell_type_grouped", groups=[tissue], show=False, size=10, alpha=0.7, ax=ax, legend_loc="none")
    
    # Set title for each subplot with the tissue name
    ax.set_title(tissue)

# Turn off unused axes if there are any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout for better spacing
plt.tight_layout()

# Show the combined grid of plots
plt.show()

### Generate_initializations matrices for Phi and Psi

In [ ]:
rho_hat = splice_adata.layers["junc_ratio"]

# Generate multiple initializations
psi_initializations, phi_initializations = wayp.generate_initializations(rho_hat, waypoints_dict, metacell_dicts, epsilon=0.001)

In [ ]:
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

In [ ]:
# Loop through the waypoints_dict and corresponding initializations
for i, n_waypoints in enumerate(waypoints_dict.keys()):

    print(f"Adding waypoint based initializations to anndata for {n_waypoints} waypoints!")

    # Extract the corresponding psi and phi initializations
    psi = psi_initializations[i]
    phi = phi_initializations[i]

    # Convert psi and phi to torch tensors if needed
    psi = torch.tensor(psi)
    phi = torch.tensor(phi)

    # Convert to NumPy arrays if psi and phi are torch tensors (just in case)
    if isinstance(psi, torch.Tensor):
        psi = psi.cpu().numpy()  # Convert to NumPy array
    if isinstance(phi, torch.Tensor):
        phi = phi.cpu().numpy()  # Convert to NumPy array

    # Store psi in `adata.varm` and phi in `adata.obsm` with keys based on the number of waypoints
    splice_adata.varm[f'psi_init_{n_waypoints}_waypoints'] = psi  # Store psi with name 'psi_init_{n_waypoints}_waypoints'
    splice_adata.obsm[f'phi_init_{n_waypoints}_waypoints'] = phi  # Store phi with name 'phi_init_{n_waypoints}_waypoints'


In [ ]:
# Convert the 'junc_ratio' layer to a CSR matrix before saving
if isinstance(splice_adata.layers['junc_ratio'], coo_matrix):
    splice_adata.layers['junc_ratio'] = splice_adata.layers['junc_ratio'].tocsr()

In [ ]:
print(f"The number of cells in the splicing dataset is {splice_adata.shape[0]}")
print(f"The number of cells in the expression dataset is {adata_reordered.shape[0]}")

### Save splice_adata Anndata object

In [ ]:
# Original file path
original_path = input_file  # Input your original file path

# Extract the directory and filename
base_directory = os.path.dirname(original_path)
original_filename = os.path.basename(original_path)

# Get the current date and time as a string (format: YYYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Replace the date and timestamp part with "with_initializations" and append the new timestamp
filename_parts = original_filename.split('_')

# Conditionally add "brain_only" to the file name if brain_only is True
if brain_only:
    new_filename = f"{'_'.join(filename_parts[:-2])}_with_initializations_brain_only_{timestamp}.h5ad"
else:
    new_filename = f"{'_'.join(filename_parts[:-2])}_with_initializations_{timestamp}.h5ad"

# Create the new full file path
new_file_path = os.path.join(base_directory, new_filename)

# Save the AnnData object with the new filename
splice_adata.write_h5ad(new_file_path, compression='gzip')

print(f"AnnData saved as {new_file_path}")

### Save Gene Expression based Anndata object! 

In [ ]:
from datetime import datetime

# Step 1: Get the current date in the desired format
current_date = datetime.now().strftime('%Y-%m-%d')

# Step 2: Format the file path with the current date
new_file_path = f"/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/TMS_Anndata_GeneExpression_{current_date}.h5ad"

# Step 3: Save the anndata object with gzip compression
adata_reordered.write_h5ad(new_file_path, compression='gzip')

print(f"Gene expression Anndata object successfully saved to {new_file_path}")

In [ ]:
splice_adata.var.annotation_status.value_counts()